# libero_task0: data exploration
Sections: 1. Setup, 2. Train/val split, 3. Action statistics, 4. Batch check and cache

## 1. Setup and dataset loading

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split, DataLoader
from torchvision.transforms.v2 import functional as F

from lerobot.datasets import video_utils
from lerobot.datasets.lerobot_dataset import LeRobotDataset

os.environ["HF_HUB_OFFLINE"] = "1"
root = os.path.expanduser("~/lerobot/data/libero_task0")

In [ ]:
eps = json.load(open(f"{root}/task0_episodes.json"))
ds = LeRobotDataset("lerobot/libero", root=root, episodes=eps)

## 2. Train / val split (by episode, seed=42)

In [ ]:
eps = torch.stack(list(ds.hf_dataset["episode_index"]))
unique_eps = torch.unique(eps)
g = torch.Generator().manual_seed(42)
train_eps, val_eps = random_split(unique_eps, [0.75, 0.25], generator=g)

with open(f"{root}/split.json", "w") as f:
    json.dump({"train": unique_eps[train_eps.indices].tolist(),
               "val": unique_eps[val_eps.indices].tolist()}, f)

In [ ]:
resize = lambda x: F.resize(x, (128, 128), antialias=True)

train_ds = LeRobotDataset("lerobot/libero", root=root, episodes=unique_eps[train_eps.indices].tolist(), image_transforms=resize)
val_ds = LeRobotDataset("lerobot/libero", root=root, episodes=unique_eps[val_eps.indices].tolist(), image_transforms=resize)

train_loader = DataLoader(train_ds, batch_size=32, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, num_workers=4, pin_memory=True)

In [ ]:
assert len(train_eps) + len(val_eps) == 44
assert len(set(train_eps) & set(val_eps)) == 0
assert len(train_ds) + len(val_ds) == 6867

## 3. Action statistics (train only)

In [ ]:
a = torch.stack(list(train_ds.hf_dataset['action']))
std_a = a.std(dim=0)
mean_a = a.mean(dim=0)
with open(os.path.expanduser(f"{root}/action_stats.json"), 'w') as f:
    json.dump({'mean_a':mean_a.tolist(), 'std_a':std_a.tolist()},f)

In [ ]:
with open(f"{root}/action_stats.json") as f:
    stats = json.load(f)
mean = torch.tensor(stats["mean_a"])
std = torch.tensor(stats["std_a"])

## 4. Normalize, cache to disk, and wrap in a Dataset

In [ ]:
video_utils._default_decoder_cache.clear()
cache_train = {'action':[],'observation.images.image':[]}
for batch in train_loader:
    cache_train['action'].append((batch['action']- mean) / std)
    cache_train['observation.images.image'].append(torch.round(batch['observation.images.image'] * 255).to(torch.uint8))
video_utils._default_decoder_cache.clear()
cache_val = {'action':[],'observation.images.image':[]}
for batch in val_loader:
    cache_val['action'].append((batch['action']- mean) / std)
    cache_val['observation.images.image'].append(torch.round(batch['observation.images.image'] * 255).to(torch.uint8))

In [ ]:
path = os.path.expanduser(f"{root}/cache_train_128.pt")
torch.save({
    "act": torch.cat(cache_train['action'], dim=0),
    "img": torch.cat(cache_train['observation.images.image'], dim=0)
}, path)
path = os.path.expanduser(f"{root}/cache_val_128.pt")
torch.save({
    "act": torch.cat(cache_val['action'], dim=0),
    "img": torch.cat(cache_val['observation.images.image'], dim=0)
}, path)

In [ ]:
path = os.path.expanduser(f"{root}/cache_train_128.pt")
cache_train = torch.load(path)
path = os.path.expanduser(f"{root}/cache_val_128.pt")
cache_val = torch.load(path)

In [ ]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, root, train=True):
        self.is_train = bool(train)

        if self.is_train:
            path = os.path.expanduser(os.path.join(root, "cache_train_128.pt"))
        else:
            path = os.path.expanduser(os.path.join(root, "cache_val_128.pt"))

        data = torch.load(path)
        self.acts = data["act"]
        self.imgs = data["img"]
    def __len__(self):
        return len(self.acts)

    def __getitem__(self, idx):
        return {'act':torch.as_tensor(self.acts[idx]), 'img':torch.as_tensor(self.imgs[idx]/255,dtype=torch.float32)}
        
ds_train = CustomDataset(root=root,train=1)
ds_val = CustomDataset(root=root,train=0)
train_loader = DataLoader(ds_train,batch_size=64, shuffle=True)
val_loader = DataLoader(ds_val,batch_size=64, shuffle=False)
batch = next(iter(train_loader))

In [ ]:
plt.imshow(cache_val['img'][50].permute(1, 2, 0).numpy()[:, :, ::])
plt.show()